## Intro

#### Install and imports

In [1]:
#pip install scikit-learn

In [2]:
import os
import pandas as pd
import sklearn
from sklearn.metrics import mean_absolute_error
import numpy as np

#### Variables to be analyzed

In [3]:
variables = ['Size', 'PdI', 'ZP', 'EE', 'DL']

#### *Monte Carlo* parameters

In [4]:
n_iterations = 200 # numbner of iterations (simulations) for each noise level 
sigmas = [0.02, 0.05, 0.1] # Noise levels as a percentage values (e.g., 2%, 5%, 10%)
epsilon = 1e-8

## Round 1

### Route to the csv file and read it

In [5]:
#Route to the file

BASE_DIR = os.getcwd()

file_path = os.path.join(
    BASE_DIR,
    "Pred_obs_R1.csv",
)

#Read the CSV file
DE_round1 = pd.read_csv(file_path, sep=";")
DE_round1 = DE_round1.dropna(how="all") # Remove rows where all values are NaN

### Metrics calculations (without "noise")

In [6]:
mae_results1 =  []
mre_results1 = []

for var in variables:
    
    y_obs = pd.to_numeric(DE_round1[f"{var}_obs"], errors='coerce')
    y_pred = pd.to_numeric(DE_round1[f"{var}_pred"], errors='coerce')

    df_temp = pd.DataFrame({"obs": y_obs, "pred": y_pred}).dropna()

    y_obs = df_temp["obs"].values
    y_pred = df_temp["pred"].values

    #MAE calculation
    mae = mean_absolute_error(y_obs, y_pred)
    
    #MRE calculation
    mre = np.mean(np.abs((y_obs - y_pred) / (np.abs(y_obs) + epsilon))) * 100

    mae_results1.append([var, mae])
    mre_results1.append([var, mre])

#Create dataframes for MAE and MRE results
mae_df1 = pd.DataFrame(mae_results1, columns=["Variable", "MAE"]).round(2)
mre_df1 = pd.DataFrame(mre_results1, columns=["Variable", "MRE_%"]).round(2)    

#Merge metrics dataframes
metrics_wo_noiseR1 = pd.merge(mae_df1, mre_df1, on="Variable")

#### Show and print results

In [7]:
#Show results
print("\nMetrics without noise R1:")
print(metrics_wo_noiseR1)

# Save results
output_path = os.path.join(BASE_DIR, "Metrics_wo_noiseR1.csv")
metrics_wo_noiseR1.to_csv(output_path, index=False)


Metrics without noise R1:
  Variable    MAE   MRE_%
0     Size  61.64  123.50
1      PdI   0.04   38.10
2       ZP   0.77  203.70
3       EE  32.12   46.02
4       DL   7.32  238.40


### Metrics calculations (with "noise" *Monte Carlo*)

In [8]:
mae_noise_results = []
mre_noise_results = []

for sigma_factor in sigmas:
    
    for var in variables:

        #Convert data to numeric and drop NaN values  
        y_obs = pd.to_numeric(DE_round1[f"{var}_obs"], errors='coerce')
        y_pred = pd.to_numeric(DE_round1[f"{var}_pred"], errors='coerce')

        df_temp = pd.DataFrame({"obs": y_obs, "pred": y_pred}).dropna()

        y_obs = df_temp["obs"].values
        y_pred = df_temp["pred"].values

        sigma = sigma_factor * np.mean(np.abs(y_obs))

        mae_list = []
        mre_list = [] 

        #Monte Carlo simulation
        for i in range(n_iterations):

            noise = np.random.normal(0, sigma, size=len(y_obs))
            y_noisy = y_obs + noise

            #MAE
            mae = mean_absolute_error(y_noisy, y_pred)
            mae_list.append(mae)

            # MRE (%)
            mre = np.mean(np.abs((y_noisy - y_pred) / (np.abs(y_noisy) + epsilon))) * 100
            mre_list.append(mre)

        mae_noise_results.append([
            var, #analyzed variable
            sigma_factor, #noise level
            np.mean(mae_list), #Mean MAE
            np.std(mae_list) #Standard deviation of MAE
        ])

        mre_noise_results.append([
            var, #analyzed variable
            sigma_factor, #noise level
            np.mean(mre_list), #Mean MRE
            np.std(mre_list)  #Standard deviation of MRE
        ])

#Create dataframes for MAE and MRE results with noise
##MAE dataframe
mae_noise_df = pd.DataFrame(
    mae_noise_results,
    columns=["Variable", "Sigma_%", "MAE_mean", "MAE_std"]
).round(2)

##MRE dataframe
mre_noise_df = pd.DataFrame(
    mre_noise_results,
    columns=["Variable", "Sigma_%", "MRE_mean", "MRE_std"]
).round(2)

#Merge MAE and MRE dataframes
metrics_noiseR1 = pd.merge(mae_noise_df, mre_noise_df, on=["Variable", "Sigma_%"])

#### Show and print results

In [9]:
print("\nMetrics with noise R1 (different levels):")
print(metrics_noiseR1)

output_path = os.path.join(BASE_DIR, "Metrics_noise_R1.csv")
metrics_noiseR1.to_csv(output_path, index=False)


Metrics with noise R1 (different levels):
   Variable  Sigma_%  MAE_mean  MAE_std  MRE_mean  MRE_std
0      Size     0.02     61.63     0.45    123.91     5.71
1       PdI     0.02      0.04     0.00     38.36     1.17
2        ZP     0.02      0.77     0.00    203.75     1.05
3        EE     0.02     32.20     0.45     46.25     0.93
4        DL     0.02      7.32     0.02    238.58     2.16
5      Size     0.05     61.58     1.28    128.43    23.11
6       PdI     0.05      0.04     0.00     38.48     3.01
7        ZP     0.05      0.77     0.01    203.83     2.61
8        EE     0.05     32.17     1.16     46.49     2.42
9        DL     0.05      7.32     0.05    239.21     5.96
10     Size     0.10     61.77     2.50    202.96   590.10
11      PdI     0.10      0.04     0.00     40.99     6.47
12       ZP     0.10      0.77     0.01    206.14     5.71
13       EE     0.10     32.50     2.32     48.08     5.27
14       DL     0.10      7.31     0.10    240.81    11.57


## Round 2

### Route to the csv file and read it

In [10]:
#Route to the file

BASE_DIR = os.getcwd()

file_path = os.path.join(
    BASE_DIR,
    "Pred_obs_R2.csv",
)

#Read the CSV file
DE_round2 = pd.read_csv(file_path, sep=";")
DE_round2 = DE_round2.dropna(how="all") # Remove rows where all values are NaN

### Metrics calculations (without "noise")

In [15]:
mae_results2 =  []
mre_results2 = []

for var in variables:
    
    y_obs = pd.to_numeric(DE_round2[f"{var}_obs"], errors='coerce')
    y_pred = pd.to_numeric(DE_round2[f"{var}_pred"], errors='coerce')

    df_temp = pd.DataFrame({"obs": y_obs, "pred": y_pred}).dropna()

    y_obs = df_temp["obs"].values
    y_pred = df_temp["pred"].values

    #MAE calculation
    mae = mean_absolute_error(y_obs, y_pred)
    
    #MRE calculation
    mre = np.mean(np.abs((y_obs - y_pred) / (np.abs(y_obs) + epsilon))) * 100

    mae_results2.append([var, mae])
    mre_results2.append([var, mre])

#Create dataframes for MAE and MRE results
mae_df2 = pd.DataFrame(mae_results2, columns=["Variable", "MAE"]).round(2)
mre_df2 = pd.DataFrame(mre_results2, columns=["Variable", "MRE_%"]).round(2)    

#Merge metrics dataframes
metrics_wo_noiseR2 = pd.merge(mae_df2, mre_df2, on="Variable"). round(2)

#### Show and print results

In [16]:
#Show results
print("\nMetrics without noise R2:")
print(metrics_wo_noiseR2)

# Save results
output_path = os.path.join(BASE_DIR, "Metrics_wo_noiseR2.csv")
metrics_wo_noiseR2.to_csv(output_path, index=False)


Metrics without noise R2:
  Variable    MAE         MRE_%
0     Size  65.68  1.296400e+02
1      PdI   0.12  4.324000e+01
2       ZP   0.80  5.745600e+02
3       EE  55.10  2.040667e+11
4       DL  11.19  2.353333e+10


### Metrics calculations (with "noise" *Monte Carlo*)

In [13]:
mae_noise_results2 = []
mre_noise_results2 = []

for sigma_factor in sigmas:
    
    for var in variables:

        #Convert data to numeric and drop NaN values  
        y_obs = pd.to_numeric(DE_round2[f"{var}_obs"], errors='coerce')
        y_pred = pd.to_numeric(DE_round2[f"{var}_pred"], errors='coerce')

        df_temp = pd.DataFrame({"obs": y_obs, "pred": y_pred}).dropna()

        y_obs = df_temp["obs"].values
        y_pred = df_temp["pred"].values

        sigma = sigma_factor * np.mean(np.abs(y_obs))

        mae_list = []
        mre_list = [] 

        #Monte Carlo simulation
        for i in range(n_iterations):

            noise = np.random.normal(0, sigma, size=len(y_obs))
            y_noisy = y_obs + noise

            #MAE
            mae = mean_absolute_error(y_noisy, y_pred)
            mae_list.append(mae)

            # MRE (%)
            mre = np.mean(np.abs((y_noisy - y_pred) / (np.abs(y_noisy) + epsilon))) * 100
            mre_list.append(mre)

        mae_noise_results2.append([
            var, #analyzed variable
            sigma_factor, #noise level
            np.mean(mae_list), #Mean MAE
            np.std(mae_list) #Standard deviation of MAE
        ])

        mre_noise_results2.append([
            var, #analyzed variable
            sigma_factor, #noise level
            np.mean(mre_list), #Mean MRE
            np.std(mre_list)  #Standard deviation of MRE
        ])

#Create dataframes for MAE and MRE results with noise
##MAE dataframe
mae_noise_df = pd.DataFrame(
    mae_noise_results2,
    columns=["Variable", "Sigma_%", "MAE_mean", "MAE_std"]
).round(2)

##MRE dataframe
mre_noise_df = pd.DataFrame(
    mre_noise_results2,
    columns=["Variable", "Sigma_%", "MRE_mean", "MRE_std"]
).round(2)

#Merge MAE and MRE dataframes
metrics_noiserR2 = pd.merge(mae_noise_df, mre_noise_df, on=["Variable", "Sigma_%"])

#### Show and print results

In [14]:
print("\nMetrics with noise R2 (different levels):")
print(metrics_noiserR2)

output_path = os.path.join(BASE_DIR, "Metrics_noise_R2.csv")
metrics_noiserR2.to_csv(output_path, index=False)


Metrics with noise R2 (different levels):
   Variable  Sigma_%  MAE_mean  MAE_std  MRE_mean   MRE_std
0      Size     0.02     65.70     0.41    130.46      4.72
1       PdI     0.02      0.12     0.00     43.40      1.26
2        ZP     0.02      0.80     0.00    575.22      7.89
3        EE     0.02     55.13     0.32  11681.54  37525.03
4        DL     0.02     11.19     0.01  47941.89  90561.20
5      Size     0.05     65.61     0.88    132.00     11.98
6       PdI     0.05      0.12     0.00     44.69      3.25
7        ZP     0.05      0.80     0.00    577.45     19.49
8        EE     0.05     55.13     0.78   8151.91  38030.90
9        DL     0.05     11.19     0.02  21029.98  45964.23
10     Size     0.10     65.61     1.82    148.97     96.69
11      PdI     0.10      0.12     0.00     54.22     33.28
12       ZP     0.10      0.80     0.00    591.92     42.80
13       EE     0.10     55.32     1.52   2314.80   5300.57
14       DL     0.10     11.19     0.03  13626.92  56385.